# Integration of Hamiltonian dynamics

We study $\dot q=p$, $\dot p=-V'(q)$ and $H(q,p)=p^2/2+V(q)$ with $M=1$.

Potentials: $V(q)=q^2/2$ and $V(q)=(q^2-1)^2$. The goal is to observe the Hamiltonian over long time for several step sizes $\Delta t$.

## Conservation of the Hamiltonian

By the chain rule,

$$\frac{dH}{dt}=V'(q)\dot q+p\dot p=V'(q)p-pV'(q)=0.$$

The Hamiltonian is therefore constant for the exact dynamics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

def V_quadratic(q):
    return 0.5 * q**2

def grad_quadratic(q):
    return q

def V_double_well(q):
    return (q**2 - 1.0)**2

def grad_double_well(q):
    return 4.0 * q * (q**2 - 1.0)

def H(q, p, potential):
    return 0.5 * p**2 + potential(q)


## Numerical schemes

Symplectic Euler A: $p_{n+1}=p_n-\Delta t V'(q_n)$ then $q_{n+1}=q_n+\Delta t p_{n+1}$.

Verlet: $p_{n+1/2}=p_n-\Delta t V'(q_n)/2$, $q_{n+1}=q_n+\Delta t p_{n+1/2}$, then $p_{n+1}=p_{n+1/2}-\Delta t V'(q_{n+1})/2$.

In [ ]:
def symplectic_euler(q0, p0, dt, T, grad_V):
    n = int(round(T / dt))
    t = dt * np.arange(n + 1)
    q, p = np.empty(n + 1), np.empty(n + 1)
    q[0], p[0] = q0, p0
    for k in range(n):
        p[k + 1] = p[k] - dt * grad_V(q[k])
        q[k + 1] = q[k] + dt * p[k + 1]
    return t, q, p

def verlet(q0, p0, dt, T, grad_V):
    n = int(round(T / dt))
    t = dt * np.arange(n + 1)
    q, p = np.empty(n + 1), np.empty(n + 1)
    q[0], p[0] = q0, p0
    for k in range(n):
        p_half = p[k] - 0.5 * dt * grad_V(q[k])
        q[k + 1] = q[k] + dt * p_half
        p[k + 1] = p_half - 0.5 * dt * grad_V(q[k + 1])
    return t, q, p


## Quadratic potential: long-time evolution

For $V(q)=q^2/2$, we have $\ddot q+q=0$, so $q(t)=q_0\cos(t)+p_0\sin(t)$ and $p(t)=-q_0\sin(t)+p_0\cos(t)$. Thus $H(t)=H(0)$ exactly.

For symplectic Euler and Verlet, the numerical energy stays bounded and oscillates if $0<\Delta t<2$. Symplectic Euler has an energy error of order $O(\Delta t)$, while Verlet is order 2 with an error of order $O(\Delta t^2)$. For $\Delta t\geq 2$, the quadratic case becomes unstable.

In [ ]:
q0, p0, T = 1.0, 0.0, 2000.0
dts = [0.1, 0.2, 0.4, 1.0]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for dt in dts:
    t, qe, pe = symplectic_euler(q0, p0, dt, T, grad_quadratic)
    t, qv, pv = verlet(q0, p0, dt, T, grad_quadratic)
    ax[0].plot(t, H(qe, pe, V_quadratic) - H(qe[0], pe[0], V_quadratic), label=f'dt={dt}')
    ax[1].plot(t, H(qv, pv, V_quadratic) - H(qv[0], pv[0], V_quadratic), label=f'dt={dt}')
ax[0].set_title('Symplectic Euler: H(t)-H(0)')
ax[1].set_title('Verlet: H(t)-H(0)')
for a in ax:
    a.set_xlabel('t')
    a.set_ylabel('energy variation')
    a.legend()
plt.tight_layout()


## Double-well potential

We repeat the study with $V(q)=(q^2-1)^2$, which has two minima at $q=\pm1$.

In [ ]:
q0, p0, T = 0.0, 1.0, 500.0
dts = [0.02, 0.05, 0.1]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for dt in dts:
    t, qe, pe = symplectic_euler(q0, p0, dt, T, grad_double_well)
    t, qv, pv = verlet(q0, p0, dt, T, grad_double_well)
    He = H(qe, pe, V_double_well)
    Hv = H(qv, pv, V_double_well)
    ax[0].plot(t, He - He[0], label=f'Euler dt={dt}')
    ax[0].plot(t, Hv - Hv[0], '--', label=f'Verlet dt={dt}')
    ax[1].plot(qv, pv, label=f'dt={dt}')
ax[0].set_title('Double well: variation of H')
ax[0].set_xlabel('t')
ax[0].set_ylabel('H(t)-H(0)')
ax[0].legend(ncol=2)
ax[1].set_title('Double well: phase portrait')
ax[1].set_xlabel('q')
ax[1].set_ylabel('p')
ax[1].legend()
plt.tight_layout()


## Conclusion

The exact dynamics conserves the Hamiltonian. The symplectic schemes limit the energy drift over long times; Verlet is more accurate than symplectic Euler because it is order 2.